In [ ]:
BATCH = 1  



In [8]:
engine = create_engine("mysql+pymysql://root:root@dq_mariadb:3306/bronze")



In [11]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("mysql+pymysql://root:root@dq_mariadb:3306/bronze")
pd.read_sql("SHOW DATABASES;", con=engine)



,Database
0,bronze
1,cif
2,crm
3,crm_cible
4,dq
5,gold
6,information_schema
7,mysql
8,performance_schema
9,silver


In [12]:
contrat = pd.read_sql("""
SELECT souscripteur_id, date_debut, date_fin, date_derniere_cotisation
FROM bronze.crm_contrat
""", con=engine)

for c in ["date_debut", "date_fin", "date_derniere_cotisation"]:
    contrat[c] = pd.to_datetime(contrat[c], errors="coerce")

ref = pd.Timestamp("2024-12-31")  # fecha de referencia
cotisation = ref - pd.Timedelta(days=90)  # últimos 3 meses

# Definiciones
en_cours = (contrat.date_debut <= ref) & (contrat.date_fin >= ref)
n_commerciale = contrat.souscripteur_id.nunique()
n_sinistres = contrat[en_cours].souscripteur_id.nunique()
n_risques = contrat[en_cours & (contrat.date_derniere_cotisation >= cotisation)].souscripteur_id.nunique()

# Tableau des écarts
ecarts = pd.DataFrame({
    "definition": ["Commerciale", "Sinistres", "Risques"],
    "nb": [n_commerciale, n_sinistres, n_risques]
})
ecarts["rapport"] = ecarts.nb / ecarts.nb.max()
ecarts


,definition,nb,rapport
0,Commerciale,0,NaN
1,Sinistres,0,NaN
2,Risques,0,NaN
